# web_agent — GOLD dataset run (Kaggle)

Real, leak-safe gold data (before+after images + task). Separate from the synthetic notebook — the synthetic 70k pipeline is untouched. **v14: all 4 pillars trained** — confidence, memory, and recovery_outcome (masked to attempted rows). Judge by **outcome_mcc** on the gold TEST split.

In [1]:
# 1. Clone the package + install deps, make it importable (re-run safe)
import os, sys
REPO = "https://github.com/Kiyas-Mahmud/webagent.git"
ROOT = "/kaggle/working/webagent"
SRC = f"{ROOT}/src"
if not os.path.isdir(ROOT):
    !git clone -b Code {REPO} {ROOT}
else:
    !cd {ROOT} && git pull --ff-only
%cd {ROOT}
!pip install -q -U "transformers>=4.49" peft bitsandbytes accelerate scikit-learn
for m in [k for k in list(sys.modules) if k == "web_agent" or k.startswith("web_agent.")]:
    del sys.modules[m]
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import web_agent
print("web_agent ready ->", list(web_agent.__path__))

Cloning into '/kaggle/working/webagent'...
remote: Enumerating objects: 420, done.
remote: Counting objects: 100% (420/420), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 420 (delta 194), reused 324 (delta 118), pack-reused 0 (from 0)
Receiving objects: 100% (420/420), 535.56 KiB | 9.74 MiB/s, done.
Resolving deltas: 100% (194/194), done.
/kaggle/working/webagent
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sklearn-compat 0.1.5 requires sciki

In [2]:
# 2. GPU + gold data path (auto-detected, slug-proof)
import os, glob
!nvidia-smi -L
print("input dirs:", os.listdir("/kaggle/input"))
hits = glob.glob("/kaggle/input/**/split_train.json", recursive=True)
assert hits, "WebGoldData not attached: right panel -> Add Input -> WebGoldData"
GOLD_PATH = os.path.dirname(hits[0])
print("GOLD_PATH =", GOLD_PATH)
print("splits:", [f for f in os.listdir(GOLD_PATH) if f.endswith(".json")])

GPU 0: Tesla T4 (UUID: GPU-5c1b4438-02c8-77fc-dcb6-e16d8620b215)
GPU 1: Tesla T4 (UUID: GPU-01f9ba43-ac62-496c-d8b9-1df8e5a7c947)
input dirs: ['datasets']
GOLD_PATH = /kaggle/input/datasets/kiyasmahmud/webgoldv3/gold_v14
splits: ['split_test.json', 'gold_export_summary.json', 'split_train.json', 'split_val.json', 'blank_filter_summary.json', 'merge_summary.json']


In [3]:
# 3. Config (gold) + processor + one gold batch
import torch
from transformers import AutoProcessor
from web_agent.config import load_config
from web_agent.utils.seed import set_seed
from web_agent.data.gold_dataloader import build_gold_dataloader, load_gold_split

set_seed(42)
cfg = load_config("configs/backbones/qwen2vl_2b_gold.yaml")
cfg["data"]["root"] = GOLD_PATH
cfg["data"]["num_workers"] = 0

bb = cfg["backbone"]
processor = AutoProcessor.from_pretrained(
    bb["vlm_model"], min_pixels=bb["min_pixels"], max_pixels=bb["max_pixels"])

train = load_gold_split(cfg, "train")
print("gold train rows:", len(train))
loader = build_gold_dataloader(cfg, "train", processor, records=train,
                               limit=8, batch_size=4, num_workers=0)
batch = next(iter(loader))
print("batch keys:", list(batch.keys()))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f"  {k:22} {tuple(v.shape)}  {v.dtype}")
print("outcome labels in batch:", batch["label_outcome"].tolist())

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

gold train rows: 3561
batch keys: ['bbox', 'bbox_mask', 'borrowed', 'label_outcome', 'label_failtype', 'label_action', 'label_recovery', 'label_memory', 'label_confidence', 'label_recovery_success', 'original_task_id', 'input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'mm_token_type_ids']
  bbox                   (4, 4)  torch.float32
  bbox_mask              (4, 1)  torch.float32
  borrowed               (4, 1)  torch.float32
  label_outcome          (4,)  torch.int64
  label_failtype         (4,)  torch.int64
  label_action           (4,)  torch.int64
  label_recovery         (4,)  torch.int64
  label_memory           (4, 1)  torch.float32
  label_confidence       (4, 1)  torch.float32
  label_recovery_success (4, 1)  torch.float32
  input_ids              (4, 542)  torch.int64
  attention_mask         (4, 542)  torch.int64
  pixel_values           (8064, 1176)  torch.float32
  image_grid_thw         (8, 3)  torch.int64
  mm_token_type_ids      (4, 542)  torch.int64
ou

In [4]:
# 4. Build model + gold class-weighted loss (v14: all 4 pillars ON)
from web_agent.models.model import WebAgentModel
from web_agent.models.loss import CombinedLoss
from web_agent.data.gold_dataloader import gold_class_weights
from web_agent.data.gold_dataset import view

model = WebAgentModel(cfg)
print("VLM hidden dim D =", model.encoder.hidden_dim, "| pooling =", cfg["backbone"]["pooling"])
device = "cuda"
for m in (model.adapter, model.failure_head, model.action_head,
          model.memory_head, model.recovery_outcome_head):
    m.to(device)

aw, fw, ow = gold_class_weights(train)
# recovery_success pos_weight from the ATTEMPTED rows only (True/False; nulls ignored).
rs_vals = [view(r)[1].get("recovery_success") for r in train]
pos = sum(x is True for x in rs_vals); neg = sum(x is False for x in rs_vals)
rsw = torch.tensor([min(neg / max(pos, 1), 5.0)])
print("action w :", [round(x,2) for x in aw.tolist()], "(6 classes incl PRESS_KEY)")
print("failtype w:", [round(x,2) for x in fw.tolist()])
print("outcome w:", [round(x,2) for x in ow.tolist()], "(capped)")
print(f"recovery pos_weight: {float(rsw):.2f}  (attempted: True {pos} / False {neg})")
loss_fn = CombinedLoss(cfg, action_class_weights=aw.to(device),
                       failtype_class_weights=fw.to(device),
                       outcome_class_weights=ow.to(device),
                       recovery_success_pos_weight=rsw.to(device)).to(device)
print("head loss weights -> confidence:", cfg["loss"]["confidence"],
      "| memory:", cfg["loss"]["memory_flag"],
      "| recovery_outcome:", cfg["loss"]["recovery_outcome"], "(all 4 pillars ON)")
print("trainable params:", f"{sum(p.numel() for p in model.trainable_parameters()):,}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290
VLM hidden dim D = 1536 | pooling = last
action w : [0.76, 1.05, 1.14, 1.02, 1.02, 1.12] (6 classes incl PRESS_KEY)
failtype w: [0.53, 1.13, 0.93, 6.74]
outcome w: [1.12, 1.0] (capped)
recovery pos_weight: 3.33  (attempted: True 142 / False 473)
head loss weights -> confidence: 0.05 | memory: 0.1 | recovery_outcome: 0.09 (all 4 pillars ON)
trainable params: 20,442,143


In [5]:
# 5. SMOKE — one batch, loss finite, all 4 pillars training
model.train()
b = next(iter(loader))
with torch.autocast("cuda", dtype=torch.float16):
    preds = model(b)
    terms = loss_fn(preds, {k: (v.to(device) if torch.is_tensor(v) else v)
                            for k, v in b.items()})
print("loss terms:", {k: round(float(v.detach()), 4) for k, v in terms.items()})
assert torch.isfinite(terms["total"]), "loss NaN/inf - STOP"
# v14: confidence + memory + recovery_outcome all trained. (recovery_outcome may be 0 in a
# batch with no attempted-recovery rows -- that's fine, it fires when such rows appear.)
print("head terms -> confidence:", round(float(terms["confidence"].detach()), 4),
      "| memory:", round(float(terms["memory"].detach()), 4),
      "| recovery_outcome:", round(float(terms["recovery_outcome"].detach()), 4))
print("contrastive non-zero:", float(terms["contrastive"].detach()) != 0.0)
print("peak GPU (GB):", round(torch.cuda.max_memory_allocated()/1e9, 2))
print("SMOKE PASS")

loss terms: {'outcome': 0.725, 'failtype': 2.0747, 'action': 1.673, 'recovery': 1.7048, 'bbox': 0.1428, 'memory': 0.4665, 'recovery_outcome': 0.0, 'confidence': 0.0688, 'calibration': 0.4491, 'contrastive': 0.9466, 'total': 1.062}
head terms -> confidence: 0.0688 | memory: 0.4665 | recovery_outcome: 0.0
contrastive non-zero: True
peak GPU (GB): 5.09
SMOKE PASS


In [6]:
# 6. TRAIN on gold + final eval on the gold TEST split (all 4 pillars)
import numpy as np
from web_agent.train.trainer import Trainer, collect_predictions, compute_metrics
from web_agent.eval.metrics import accuracy, outcome_mcc

cfg["train"]["epochs"] = 5
cfg["data"]["num_workers"] = 4
train_loader = build_gold_dataloader(cfg, "train", processor, shuffle=True, num_workers=4)
val_loader   = build_gold_dataloader(cfg, "val",   processor, shuffle=False, num_workers=4)

trainer = Trainer(model, loss_fn, cfg, train_loader, val_loader, train_sampler=None)
print("training done:", trainer.fit())

test_loader = build_gold_dataloader(cfg, "test", processor, shuffle=False, num_workers=4)
p = collect_predictions(model, test_loader, device)
gold_metrics = compute_metrics(p)

# recovery_outcome: score ONLY the attempted rows (label != -1). Overwrite the raw
# (null-contaminated) recovery_outcome_acc and add a real MCC.
rot = np.asarray(p["recovery_outcome_true"]); rop = np.asarray(p["recovery_outcome_pred"])
msk = rot >= 0
gold_metrics["recovery_outcome_acc"] = accuracy(rot[msk], rop[msk]) if msk.any() else 0.0
gold_metrics["recovery_outcome_mcc"] = outcome_mcc(rot[msk], rop[msk]) if msk.any() else 0.0

print("GOLD TEST:", {k: round(v, 4) for k, v in gold_metrics.items()})
print("  HEADLINE = outcome_mcc / outcome_bal_acc / failure_macro_f1")
print(f"  ALL 4 PILLARS real. recovery_outcome scored on {int(msk.sum())} attempted rows.")

epoch 0 | step 50/112 | loss 0.548 | elapsed 39.9min | ETA 49.4min
epoch 0 | step 100/112 | loss 0.385 | elapsed 79.7min | ETA 9.6min
epoch 0: failure_f1=0.9736  failure_macro_f1=0.9722  outcome_bal_acc=0.9730  outcome_mcc=0.9448  success_recall=0.9839  failtype_acc=0.9160  action_acc=1.0000  recovery_acc=0.8371  recovery_outcome_acc=0.1335  memory_acc=0.9110  ece=0.4007  bbox_mae=0.1164
epoch 1 | step 50/112 | loss 1.926 | elapsed 39.9min | ETA 49.5min
epoch 1 | step 100/112 | loss 0.313 | elapsed 79.8min | ETA 9.6min
epoch 1: failure_f1=0.9743  failure_macro_f1=0.9731  outcome_bal_acc=0.9740  outcome_mcc=0.9466  success_recall=0.9875  failtype_acc=0.8816  action_acc=1.0000  recovery_acc=0.8413  recovery_outcome_acc=0.1419  memory_acc=0.9211  ece=0.3789  bbox_mae=0.0709
epoch 2 | step 50/112 | loss 0.495 | elapsed 39.9min | ETA 49.4min
epoch 2 | step 100/112 | loss 1.113 | elapsed 79.7min | ETA 9.6min
epoch 2: failure_f1=0.9782  failure_macro_f1=0.9773  outcome_bal_acc=0.9785  outcome

In [7]:
# 7. Save the gold-TEST result to ONE CSV -> download from the Output panel
import csv as _csv
assert "gold_metrics" in globals(), "Run cell 6 to completion first (it defines gold_metrics)."
VERSION = "gold_v14"   # <-- change per run (gold_v14, gold_v15, ...)

csv_path = f"/kaggle/working/gold_test_{VERSION}.csv"
with open(csv_path, "w", newline="") as f:
    w = _csv.writer(f)
    w.writerow(["version"] + list(gold_metrics.keys()))
    w.writerow([VERSION] + [round(v, 5) for v in gold_metrics.values()])

print("SAVED CSV:", csv_path)
print("GOLD TEST:", {k: round(v, 4) for k, v in gold_metrics.items()})
print("Download it from the right-side Output panel (/kaggle/working).")

SAVED CSV: /kaggle/working/gold_test_gold_v14.csv
GOLD TEST: {'failure_f1': 0.9708, 'failure_macro_f1': 0.9694, 'outcome_bal_acc': 0.9699, 'outcome_mcc': 0.939, 'success_recall': 0.9767, 'failtype_acc': 0.9093, 'action_acc': 1.0, 'recovery_acc': 0.8568, 'recovery_outcome_acc': 0.797, 'memory_acc': 0.9339, 'ece': 0.3388, 'bbox_mae': 0.04, 'recovery_outcome_mcc': 0.5838}
Download it from the right-side Output panel (/kaggle/working).
